# **3. Обучение бейзлайна - логистической регрессии**

Для бейзлайна была выбрана логистическая регрессия, как наиболее простая модель.

**Цель:** получить первую рабочую модель и метрику Precision@Recall>=0.7

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, precision_score, recall_score
import matplotlib.pyplot as plt
import warnings

from src.data_loader import load_train, load_events
from src.feature_engineering import build_features
from src.metrics import precision_at_recall

warnings.filterwarnings('ignore')

In [6]:
train, events = load_train(), load_events()
features = build_features(events, train, verbose=True)
features.head()

Событий в окне: 198504
Итого признаков: 46


,cookie_id,event_count,unique_items,unique_categories,unique_locations,unique_event_types,ua_length,has_headless,has_bot_pattern,n_unique_ua,...,ratio_item_view,ratio_login,ratio_photo_swipe,ratio_search_results_view,ratio_seller_page_view,has_favorite_add,has_login,has_captcha,item_view_to_photo_ratio,target
0,ck_000c95f1408dcb00,16,9,2,8,4,117.0,0,0,1,...,0.687500,0.000000,0.062500,0.187500,0.062500,0,0,0,5.500000,0
1,ck_000e8c52636e3bec,34,20,4,12,7,80.0,0,0,1,...,0.323529,0.029412,0.147059,0.264706,0.088235,1,1,0,1.833333,0
2,ck_0010e31baa4a1fb7,16,8,3,4,6,130.0,0,0,1,...,0.437500,0.062500,0.062500,0.250000,0.062500,1,1,0,3.500000,0
3,ck_0010ec3874fb5378,74,31,2,19,6,130.0,0,0,1,...,0.337838,0.013514,0.189189,0.270270,0.054054,1,1,0,1.666667,0
4,ck_001722063b94cae0,15,8,3,9,4,123.0,0,0,1,...,0.533333,0.000000,0.066667,0.333333,0.000000,0,0,0,4.000000,0


In [7]:
X = features.drop(columns=['cookie_id', 'target'])
y = features['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

In [12]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # NaN → медиана
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(
        max_iter=2000,
        class_weight='balanced',   # если дисбаланс
        random_state=42,
    )),
])

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](46,)","['event_count','unique_items','unique_categories',...,'has_login', 'has_captcha','item_view_to_photo_ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,46
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences o

In [17]:
y_scores = pipeline.predict_proba(X_val)[:, 1]

p_at_r = precision_at_recall(y_val, y_scores)
print(f"Precision@Recall>=0.7: {p_at_r:.4f}")

Precision@Recall>=0.7: 0.4038


In [18]:
from sklearn.metrics import roc_auc_score
print(f"ROC-AUC: {roc_auc_score(y_val, y_scores):.4f}")

ROC-AUC: 0.8903
